### Group Roles

- Srijitha: Prompt Engineering, BERT model implementation, notebook development, and rubric updates.
- Hemanth: Data preprocessing, cleaning support, and dataset preparation.
- Isagani and Kusuma : Review of notebook, model optimization suggestions, and final integration for submission.

### Feature Selection and Justification

We selected text-based features because fraudulent job postings are often identified through suspicious language patterns, unrealistic requirements, and misleading descriptions. Text fields provide the richest information for distinguishing fraudulent and legitimate postings.

BERT was chosen because it captures contextual meaning better than traditional text-vectorization methods. This helps the model learn semantic differences in posting content more effectively.

Final settings used:
- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16

In [18]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

In [2]:
import pandas as pd
import re

# Load data
train_df = pd.read_csv('/content/train_clean_v1.csv')
val_df = pd.read_csv('/content/val.csv')

# Fill missing values
text_cols = ['title', 'description', 'requirements', 'benefits']

for col in text_cols:
    train_df[col] = train_df[col].fillna('')
    val_df[col] = val_df[col].fillna('')

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

for col in text_cols:
    train_df[col] = train_df[col].apply(clean_text)
    val_df[col] = val_df[col].apply(clean_text)

# Combine text
train_df['full_text'] = train_df['title'] + ' ' + train_df['description'] + ' ' + train_df['requirements'] + ' ' + train_df['benefits']
val_df['full_text'] = val_df['title'] + ' ' + val_df['description'] + ' ' + val_df['requirements'] + ' ' + val_df['benefits']

train_df[['full_text']].head()

,full_text
0,inside sales consultant b2b software company p...
1,cad designer pbiwe have more than 1500 job ope...
2,microgrid systems engineer pwe have extensive ...
3,software engineer pour engineering team values...
4,messenger courier part time pthe messenger co...


In [3]:
!pip install transformers

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

In [12]:
df = pd.read_csv("train_clean_v1.csv")
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset,country,desc_len
0,Inside Sales Consultant B2B software company,"US, GA, Atlanta",Sales,35000-150000,<p>Katapult Group - We leverage technology and...,<p>Katapult Group is a global business buildin...,<p>Although this product converts extremly hig...,<p><b>Compensation:</b> 3000-5000$ per month p...,0,1,1,Full-time,Entry level,Certification,Internet,Sales,0,0,US,1907
1,Cad Designer,"US, OH, Cleveland",NaN,NaN,<p>We Provide Full Time Permanent Positions fo...,<p><b><i>(We have more than 1500 Job openings ...,Unspecified,Unspecified,0,0,0,Full-time,NaN,NaN,NaN,NaN,0,0,US,2505
2,Micro-grid Systems Engineer,"US, CA, Santa Monica",tech,NaN,<p>hello world</p>\r\n<p>talents23_ drives the...,<p>We have extensive experience in battery sto...,<ul>\r\n<li>Experience with utility interactiv...,"<p>Want to be part of a fast growing, high ene...",0,1,1,Full-time,NaN,NaN,NaN,NaN,0,0,US,699
3,Software Engineer,"US, NY, Brooklyn",Engineering,NaN,<p>Maker’s Row is an online marketplace that c...,<p>Our Engineering team values software qualit...,<ul>\r\n<li>3+ years of full-stack development...,<ul>\r\n<li>Healthcare</li>\r\n</ul><ul>\r\n<l...,0,1,1,Full-time,Associate,Bachelor's Degree,Computer Software,Engineering,0,0,US,1162
4,Messenger Courier - Part Time,"US, DC, Washington",NaN,NaN,"<p>Novitex Enterprise Solutions, formerly Pitn...",<p>The Messenger Courier will be based in our ...,<p><b>Minimum Requirements:</b></p>\r\n<ul>\r\...,Unspecified,0,1,0,Part-time,Entry level,High School or equivalent,Government Administration,Customer Service,0,0,US,1489


In [21]:
df['text'] = df['title'].fillna('') + " " + \
             df['company_profile'].fillna('') + " " + \
             df['description'].fillna('') + " " + \
             df['requirements'].fillna('')

In [22]:
texts = df['text']
labels = df['fraudulent']

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
print(df.columns)

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset', 'country', 'desc_len'],
      dtype='object')


In [14]:
labels = df['fraudulent']

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([ 0.5255, 10.3033])


In [15]:
df = df.sample(frac=0.3, random_state=42)

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [23]:
texts = df['text']
labels = df['fraudulent']

encodings = tokenize_function(texts.tolist())

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = Dataset(encodings, labels)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [25]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [26]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

/tmp/ipykernel_5980/2736003858.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [27]:
epochs = 1

model.train()

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    total_loss = 0

    for i, batch in enumerate(tqdm(train_loader)):

        if i > 200:
            break

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)


        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fct(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / (i+1)
    print(f"Training Loss: {avg_loss}")


Epoch 1/1


  0%|          | 0/164 [00:00<?, ?it/s]/tmp/ipykernel_5980/2698198337.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 164/164 [00:28<00:00,  5.66it/s]

Training Loss: 0.5312167630053874
